# 07 直接法综合实验

本 Notebook 汇总第六章的关键实验：主元选择、PLU 重构、多个右端项、SPD 分解、三对角效率、QR 正交性和病态矩阵上的残差/前向误差。


In [ ]:
import pathlib
import sys
import time

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
CHAPTER = ROOT / "chapters" / "ch06_direct_linear_systems"
SCRIPTS = CHAPTER / "scripts"
for path in [SRC, SCRIPTS]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from py_sc import (
    cholesky_factorization,
    cholesky_solve,
    classical_gram_schmidt,
    gaussian_elimination,
    gaussian_elimination_partial_pivoting,
    householder_qr,
    ldlt_factorization,
    ldlt_solve,
    lu_solve,
    modified_gram_schmidt,
    orthogonality_error,
    plu_factorization,
    thomas_algorithm,
)
from py_sc.direct_linear import relative_forward_error, relative_residual

import direct_methods as chapter_script


## 实验 1：不选主元与部分选主元


In [ ]:
no_pivot, pivot = chapter_script.pivoting_demo()
print("脚本快速检查：")
print("不选主元:", no_pivot)
print("部分选主元:", pivot)

A = np.array([[1e-16, 1.0], [1.0, 1.0]])
x_exact = np.ones(2)
b = A @ x_exact
for name, x in [("不选主元", gaussian_elimination(A, b, tol=0.0)), ("部分选主元", gaussian_elimination_partial_pivoting(A, b))]:
    print(f"{name:8s} 残差={relative_residual(A, x, b):.3e}, 前向误差={relative_forward_error(x, x_exact):.3e}")


## 实验 2：PLU 重构与多个右端项复用


In [ ]:
rng = np.random.default_rng(2026)
A = rng.normal(size=(8, 8)) + 3.0 * np.eye(8)
B = rng.normal(size=(8, 4))
plu = plu_factorization(A)
X = lu_solve(plu.L, plu.U, B, permutation=plu.permutation)
print("||PA-LU||:", np.linalg.norm(A[plu.permutation, :] - plu.L @ plu.U))
print("多个右端项相对残差:", np.linalg.norm(B - A @ X) / np.linalg.norm(B))


## 实验 3：Cholesky、LDLᵀ 与一般 LU 的比较


In [ ]:
R = rng.normal(size=(10, 10))
A_spd = R.T @ R + np.eye(10)
x_exact = rng.normal(size=10)
b = A_spd @ x_exact
L_chol = cholesky_factorization(A_spd)
x_chol = cholesky_solve(L_chol, b)
L_ldlt, D = ldlt_factorization(A_spd)
x_ldlt = ldlt_solve(L_ldlt, D, b)
plu = plu_factorization(A_spd)
x_plu = lu_solve(plu.L, plu.U, b, permutation=plu.permutation)
for name, x in [("Cholesky", x_chol), ("LDLT", x_ldlt), ("PLU", x_plu)]:
    print(f"{name:9s} 残差={relative_residual(A_spd, x, b):.3e}, 前向误差={relative_forward_error(x, x_exact):.3e}")


## 实验 4：三对角追赶法效率

运行时间有波动，这里只展示规模增长趋势。


In [ ]:
sizes = [100, 300, 700]
thomas_times = []
dense_times = []
for n in sizes:
    lower = -np.ones(n - 1)
    diagonal = 4.0 * np.ones(n)
    upper = -np.ones(n - 1)
    b = np.ones(n)
    A_dense = np.diag(diagonal) + np.diag(lower, -1) + np.diag(upper, 1)
    t0 = time.perf_counter()
    thomas_algorithm(lower, diagonal, upper, b)
    thomas_times.append(time.perf_counter() - t0)
    t0 = time.perf_counter()
    np.linalg.solve(A_dense, b)
    dense_times.append(time.perf_counter() - t0)

plt.figure(figsize=(7, 4))
plt.loglog(sizes, thomas_times, "o-", label="Thomas")
plt.loglog(sizes, dense_times, "s-", label="dense solve")
plt.xlabel("矩阵阶数")
plt.ylabel("秒")
plt.title("三对角求解与一般稠密求解的时间对比")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()


## 实验 5：QR 正交性比较


In [ ]:
A_qr = np.array([[1.0, 1.0, 1.0], [1.0, 1.0 + 1e-8, 2.0], [1.0, 2.0, 3.0], [1.0, 3.0, 5.0]])
qr_results = []
for name, factor in [("经典 GS", classical_gram_schmidt), ("修正 GS", modified_gram_schmidt), ("Householder", householder_qr)]:
    Q, R = factor(A_qr)
    qr_results.append((name, np.linalg.norm(Q @ R - A_qr), orthogonality_error(Q)))
    print(f"{name:12s} 重构误差={qr_results[-1][1]:.3e}, 正交性误差={qr_results[-1][2]:.3e}")

plt.figure(figsize=(7, 4))
plt.bar([r[0] for r in qr_results], [r[2] for r in qr_results])
plt.yscale("log")
plt.ylabel("||Q^T Q - I||_F")
plt.title("QR 方法的正交性误差")
plt.show()


## 实验 6：病态矩阵上的残差和前向误差

Hilbert 矩阵条件数很大。即使相对残差很小，前向误差也可能明显增大。


In [ ]:
def hilbert_matrix(n):
    i = np.arange(1, n + 1)
    return 1.0 / (i[:, None] + i[None, :] - 1.0)

sizes = np.arange(4, 13)
conditions = []
residuals = []
forward_errors = []
for n in sizes:
    H = hilbert_matrix(int(n))
    x_exact = np.ones(n)
    b = H @ x_exact
    x = gaussian_elimination_partial_pivoting(H, b, tol=0.0)
    conditions.append(np.linalg.cond(H))
    residuals.append(relative_residual(H, x, b))
    forward_errors.append(relative_forward_error(x, x_exact))

plt.figure(figsize=(7, 4))
plt.semilogy(sizes, residuals, "o-", label="相对残差")
plt.semilogy(sizes, forward_errors, "s-", label="相对前向误差")
plt.semilogy(sizes, np.array(conditions) / max(conditions), "--", label="条件数缩放参考")
plt.xlabel("Hilbert 矩阵阶数")
plt.ylabel("误差或缩放条件数")
plt.title("小残差不一定意味着小前向误差")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()


## 综合小结

* 高斯消元和 LU 分解本质上是同一个消元过程的两种表达。
* 部分选主元可以避免零主元并减轻小主元导致的舍入误差放大。
* 多个右端项时，矩阵分解的复用优势很明显。
* 对称正定矩阵应优先考虑 Cholesky 或 $LDL^T$。
* 三对角系统应使用追赶法，避免把结构化问题当作一般稠密问题处理。
* QR 分解需要同时检查重构误差和正交性误差。
* 条件数反映问题本身的敏感性，算法稳定性反映算法是否额外放大误差，两者不能混为一谈。
